<a href="https://colab.research.google.com/github/kaluarachchiteshan/agromaster-risk-calculation-ml/blob/05-split-labels-baselines/notebooks/05_split_labels_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 05 - Chronological split, validation folds, risk labels and baselines (proposal 4.5 - 4.7) | Group 41 AgroMaster
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

OUT_DIR = "/content/drive/MyDrive/agromaster"

# ---------------- CONFIG (from the proposal) ----------------
DEV_END  = pd.Timestamp("2023-12-31")   # development period ends here; everything after is the final test period
N_BLOCKS = 6                            # development period is cut into 6 consecutive time blocks -> 5 folds
Q_HIGH, Q_MEDIUM = 0.10, 0.30           # ratio <= 10th percentile = High, <= 30th percentile = Medium, else Low
# ------------------------------------------------------------
CLASS_NAMES = ["Low", "Medium", "High"] # encoded as 0, 1, 2

data = pd.read_parquet(os.path.join(OUT_DIR, "features.parquet"))
print(f"{len(data):,} rows | {data['date'].min().date()} to {data['date'].max().date()}")

Mounted at /content/drive
5,784,407 rows | 2001-08-04 to 2025-10-30


In [ ]:
# Assign every row to a block: 0..5 = development blocks in time order, 6 = final test, -1 = dropped
is_dev  = data["date"] <= DEV_END
crosses = is_dev & (data["target_date"] > DEV_END)      # development rows whose 7-day target falls in the test period

# Cut the development period at dates so that each block holds about the same number of rows.
# All rows that share a date go to the same block.
dev_dates = data.loc[is_dev & ~crosses, "date"]
cuts = [dev_dates.quantile(i / N_BLOCKS, interpolation="nearest") for i in range(1, N_BLOCKS)]
edges = [dev_dates.min()] + cuts + [DEV_END + pd.Timedelta(days=1)]

data["block"] = np.int8(N_BLOCKS)                        # default: final test
data.loc[is_dev, "block"] = (np.searchsorted(np.array(cuts, dtype="datetime64[ns]"), data.loc[is_dev, "date"].values, side="right")).astype("int8")
data.loc[crosses, "block"] = -1

summary = data.groupby("block")["date"].agg(rows="size", start="min", end="max")
summary.index = summary.index.map(lambda b: "dropped" if b == -1 else ("TEST" if b == N_BLOCKS else f"block {b}"))
print(summary.to_string())

data.to_parquet(os.path.join(OUT_DIR, "model_table.parquet"), compression="zstd", index=False)
json.dump({"dev_end": str(DEV_END.date()), "n_blocks": N_BLOCKS, "block_edges": [str(pd.Timestamp(e).date()) for e in edges],
           "q_high": Q_HIGH, "q_medium": Q_MEDIUM, "class_names": CLASS_NAMES},
          open(os.path.join(OUT_DIR, "splits.json"), "w"), indent=2)
print("\nSaved model_table.parquet and splits.json")

           rows      start        end
block                                
dropped    6622 2023-12-25 2023-12-31
block 0  860390 2001-08-04 2010-11-18
block 1  860784 2010-11-19 2014-06-04
block 2  860608 2014-06-05 2017-03-06
block 3  860242 2017-03-07 2019-04-25
block 4  861870 2019-04-26 2021-09-28
block 5  860821 2021-09-29 2023-12-24
TEST     613070 2024-01-01 2025-10-30

Saved model_table.parquet and splits.json


In [ ]:
# Helper functions (the same cell is reused in the modelling notebooks)
def get_fold(data, k):
    """Fold k (1..5): train on blocks 0..k-1, validate on block k.
    Training rows whose 7-day target date reaches the validation period are excluded."""
    val = data[data["block"] == k]
    train = data[(data["block"] >= 0) & (data["block"] < k) & (data["target_date"] < val["date"].min())]
    return train, val

def fit_thresholds(train):
    """Commodity-specific 10th and 30th percentiles of the target ratio, estimated on training rows only."""
    thr = train.groupby("commodity", observed=True)["ratio"].quantile([Q_HIGH, Q_MEDIUM]).unstack()
    thr.columns = ["p_high", "p_medium"]
    return thr

def make_labels(d, thr, col="ratio"):
    """0 = Low, 1 = Medium, 2 = High risk, using the thresholds of each row's commodity."""
    p_high = d["commodity"].astype(str).map(thr["p_high"]).to_numpy(dtype="float64")
    p_med  = d["commodity"].astype(str).map(thr["p_medium"]).to_numpy(dtype="float64")
    r = d[col].to_numpy(dtype="float64")
    return np.where(r <= p_high, 2, np.where(r <= p_med, 1, 0)).astype("int8")

def score(y_true, y_pred):
    """Metrics from the proposal; macro F1 and High-risk recall are the main ones."""
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0),
            "high_recall": recall_score(y_true, y_pred, labels=[2], average="macro", zero_division=0),
            "high_precision": precision_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)}

In [ ]:
# Folds, labels and the two simple baselines (validation folds only; the test period stays untouched)
rows, thr_rows = [], []
for k in range(1, N_BLOCKS):
    train, val = get_fold(data, k)
    thr = fit_thresholds(train)
    y_train, y_val = make_labels(train, thr), make_labels(val, thr)
    thr_rows.append(thr.assign(fold=k).reset_index())

    dist = pd.Series(y_val).value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)
    print(f"Fold {k}: train {len(train):>9,} rows ({train['date'].min().date()} to {train['date'].max().date()}) | "
          f"val {len(val):>9,} rows ({val['date'].min().date()} to {val['date'].max().date()}) | "
          f"val classes Low/Med/High = {dist[0]:.2f}/{dist[1]:.2f}/{dist[2]:.2f}")

    majority = np.full(len(val), pd.Series(y_train).mode()[0], dtype="int8")     # always predict the most common training class
    persistence = make_labels(val, thr, col="current_ratio")                     # today's ratio class is predicted for t+7
    rows.append({"model": "Majority class", "fold": k, **score(y_val, majority)})
    rows.append({"model": "Risk persistence", "fold": k, **score(y_val, persistence)})

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "cv_results_baselines.csv"), index=False)
pd.concat(thr_rows).to_csv(os.path.join(OUT_DIR, "cv_thresholds.csv"), index=False)

print("\nPer fold:\n" + res.round(3).to_string(index=False))
print("\nMean over the 5 folds:\n" + res.groupby("model")[["accuracy", "macro_f1", "high_recall", "high_precision"]].mean().round(3).to_string())
print("\nThresholds per commodity, fold 1 vs fold 5 (ratio at or below p_high = High risk):")
t = pd.concat(thr_rows); print(t[t["fold"].isin([1, N_BLOCKS - 1])].round(3).to_string(index=False))

Fold 1: train   856,760 rows (2001-08-04 to 2010-11-11) | val   860,784 rows (2010-11-19 to 2014-06-04) | val classes Low/Med/High = 0.67/0.21/0.11
Fold 2: train 1,716,384 rows (2001-08-04 to 2014-05-28) | val   860,608 rows (2014-06-05 to 2017-03-06) | val classes Low/Med/High = 0.68/0.20/0.11
Fold 3: train 2,575,268 rows (2001-08-04 to 2017-02-27) | val   860,242 rows (2017-03-07 to 2019-04-25) | val classes Low/Med/High = 0.72/0.19/0.09
Fold 4: train 3,434,527 rows (2001-08-04 to 2019-04-18) | val   861,870 rows (2019-04-26 to 2021-09-28) | val classes Low/Med/High = 0.73/0.18/0.09
Fold 5: train 4,297,083 rows (2001-08-04 to 2021-09-21) | val   860,821 rows (2021-09-29 to 2023-12-24) | val classes Low/Med/High = 0.71/0.20/0.09

Per fold:
           model  fold  accuracy  macro_f1  high_recall  high_precision
  Majority class     1     0.675     0.269        0.000           0.000
Risk persistence     1     0.815     0.731        0.644           0.748
  Majority class     2     0.683 